# Lab 2 — Hardware economics: the same sweep on three GPUs

**The claim you should be able to make when you finish:** *"I can tell you which
GPU to buy for a given workload and show the sweep that says so — and I can tell
you when the cheap card wins, which is more often than people expect."*

Runtime switching is the secretly great thing about Colab for this lab. The same
notebook, run on **T4 → L4 → A100**, is a hardware comparison with zero infra
work: no provisioning, no drivers, no cluster. Change the runtime type, rerun,
and the results file accumulates another row.

### The method

1. Sweep concurrency on whatever GPU is attached, and save the results to disk.
2. Switch runtime type. Rerun. Repeat.
3. Load every saved run and compare **throughput per dollar**, not throughput.
4. Check the numbers against the roofline prediction, and explain the gaps.

### The trap this lab exists to defuse

The fastest GPU is almost never the cheapest way to serve a given QPS, and the
cheapest GPU is often not viable at all because the model does not fit. Both
halves matter. A sweep that only reports tokens/s answers neither question.

In [ ]:
# Cell 1 — idempotent bootstrap. See lab 1 for the details.
REPO   = "https://github.com/lsgrep/serv.git"
BRANCH = "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("matplotlib", "pandas", "httpx")
pip("vllm")

import servlab
env = servlab.notebook_setup()

In [ ]:
# Results must outlive the runtime — you are about to switch hardware, which
# wipes the disk. Drive is the only thing that survives.
from servlab.env import mount_drive

RESULTS_DIR = "/content/drive/MyDrive/servlab" if mount_drive() else "runs"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("results ->", RESULTS_DIR)

## 1. Predict all three cards before measuring any of them

The roofline model is two lines and it explains most of what you are about to
measure:

* **Prefill is compute bound.** It is a big matmul over the whole prompt, so
  time ≈ `2 · params · tokens / FLOPS`. Cards with more TFLOPs win.
* **Decode is memory bound.** One token per sequence per step, so time ≈
  `(weights + KV) / bandwidth`. Cards with more bandwidth win, and FLOPs barely
  matter.

That is why an L4 can lose to a T4 on decode throughput despite having roughly
twice the FLOPs — L4 bandwidth (~300 GB/s) is no better than T4's (~320 GB/s).
Look for that in the measurement. It surprises people, and being able to explain
it from first principles is the point of the lab.

In [ ]:
from servlab import napkin as nk
from servlab.plots import use_style, SERIES, STATUS
import matplotlib.pyplot as plt

SPEC = nk.MODELS["qwen2.5-3b"]
CARDS = ["T4", "L4", "A100-40GB"]
CTX = 384
BATCHES = [1, 2, 4, 8, 16, 32, 64]

use_style()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for i, card in enumerate(CARDS):
    tps = [nk.decode_tokens_per_s(card, SPEC, batch=b, ctx_len=CTX) for b in BATCHES]
    axes[0].plot(BATCHES, tps, marker="o", color=SERIES[i], label=card)
    axes[1].plot(BATCHES, [nk.cost_per_million_tokens(card, t) for t in tps],
                 marker="o", color=SERIES[i], label=card)

axes[0].set_xscale("log", base=2); axes[0].set_xlabel("batch size")
axes[0].set_ylabel("output tokens/s"); axes[0].set_title("predicted decode throughput")
axes[0].legend(loc="upper left")
axes[1].set_xscale("log", base=2); axes[1].set_yscale("log")
axes[1].set_xlabel("batch size"); axes[1].set_ylabel("$ per 1M output tokens")
axes[1].set_title("predicted cost — the curve that decides")
axes[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

for card in CARDS:
    g = nk.GPUS[card]
    print(f"{card:<12} {g.mem_bw_gb_s:>6,.0f} GB/s  {g.fp16_tflops:>6,.0f} TF  "
          f"ridge {nk.ridge_point(card):>5,.0f} FLOP/byte  ${g.usd_per_hour:.2f}/h  "
          f"max seqs @ {CTX}: {nk.max_concurrent_sequences(card, SPEC, CTX):>5,.0f}")

> **The `usd_per_hour` numbers in `servlab.napkin.GPUS` are placeholders.** Edit
> them to what you are actually paying — a reserved A100 and an on-demand A100
> differ by more than the performance gap you are about to measure, so every
> cost conclusion here is downstream of that one field.

## 2. Sweep this GPU

Closed loop this time, deliberately. Lab 1 needed open loop to reproduce
overload; here we want each concurrency level to be a *stable operating point*,
so the client holds exactly N requests in flight and we read off what the server
does with them. That is the curve you would put in a capacity plan.

In [ ]:
from servlab.serve import VLLMServer
from servlab.loadgen import sweep
import json, time

MODEL = "Qwen/Qwen2.5-3B-Instruct"
MAX_MODEL_LEN = 2048
LEVELS = [1, 2, 4, 8, 16, 32, 64]

server = VLLMServer(MODEL, port=8000, max_model_len=MAX_MODEL_LEN,
                    gpu_memory_utilization=0.90, enforce_eager=True,
                    log_path="runs/lab2.log").start()

rows = sweep("http://localhost:8000", MODEL, LEVELS, mode="concurrency",
             duration=25, warmup=5, prompt_tokens=256, max_tokens=128,
             slo_ttft=1.0, slo_tpot=0.05)

card = env.gpu_name.replace(" ", "_") or "cpu"
payload = {"gpu": env.gpu_name, "vram_gb": env.vram_gb, "model": MODEL,
           "capability": list(env.capability), "when": time.strftime("%Y-%m-%d %H:%M"),
           "rows": rows}
path = f"{RESULTS_DIR}/sweep_{card}.json"
with open(path, "w") as f:
    json.dump(payload, f, indent=2)
print("saved ->", path)

In [ ]:
from servlab.plots import sweep_curves
sweep_curves(rows, title=f"{env.gpu_name} — concurrency sweep", xlabel="concurrency");

In [ ]:
# Prediction vs measurement, on the same axes. A model that is within 2x is a
# working model; the residual is where the interesting questions live.
import matplotlib.pyplot as plt
from servlab.plots import SERIES

GPU_KEY = "T4" if "T4" in env.gpu_name else ("L4" if "L4" in env.gpu_name else "A100-40GB")
pred = [nk.decode_tokens_per_s(GPU_KEY, SPEC, batch=r["level"], ctx_len=384) for r in rows]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot([r["level"] for r in rows], pred, marker="o", color=SERIES[0], label="roofline prediction")
ax.plot([r["level"] for r in rows], [r["out_tok_per_s"] for r in rows],
        marker="o", color=SERIES[1], label="measured")
ax.set_xscale("log", base=2)
ax.set_xlabel("concurrency"); ax.set_ylabel("output tokens/s")
ax.set_title("roofline vs reality")
ax.legend(loc="upper left")
plt.show()

for r, p in zip(rows, pred):
    print(f"  b={r['level']:>3}  predicted {p:>7,.0f}  measured {r['out_tok_per_s']:>7,.0f}  "
          f"ratio {r['out_tok_per_s']/p:>5.2f}")

Where the gap usually comes from, in the order worth checking:

1. **The server was not running the batch you asked for.** N concurrent clients
   is not N sequences in the engine — check `vllm:num_requests_running` during
   the sweep, not the client's concurrency setting.
2. **Prefill time is in there too.** The prediction above is decode only. At
   short outputs, prefill is a large share of the request.
3. **`--enforce-eager` costs 10-20%.** CUDA graphs were disabled to save VRAM.
4. **Bandwidth efficiency is not 100%.** The default assumes 70% of spec sheet;
   if you need more than 100% to fit the data, your model of what is being read
   is wrong — that is a real finding, not a fudge factor.

In [ ]:
server.stop()

## 3. Switch the runtime and do it again

**Runtime → Change runtime type → T4 / L4 / A100 → Save.** The session restarts
and the disk is wiped, which is why results went to Drive.

Then: rerun cell 1, the Drive cell, and the sweep cell. Roughly 15 minutes per
card, most of it the model download.

Notes per card:

* **T4 (free)** — 16 GB, no bf16, no FP8. `--dtype half` is mandatory; `servlab`
  passes it for you. A 3B fp16 model fits with room for KV; an 8B fp16 does not
  fit at all.
* **L4 (Pro)** — 24 GB, Ada. bf16 and FP8 storage work. Bandwidth is *not* much
  better than T4, so watch decode throughput fail to improve the way the FLOPs
  suggest it should.
* **A100 (Pro+)** — 40 GB, ~5x T4 bandwidth. This is where decode throughput
  finally moves. Ask whether it moved 5x, and if not, what else is binding.

In [ ]:
# Run this once you have two or more saved sweeps.
import glob, json

runs = []
for p in sorted(glob.glob(f"{RESULTS_DIR}/sweep_*.json")):
    with open(p) as f:
        runs.append(json.load(f))
print(f"{len(runs)} run(s):", ", ".join(r["gpu"] for r in runs))

In [ ]:
from servlab.plots import use_style, SERIES
import matplotlib.pyplot as plt

use_style()
fig, axes = plt.subplots(2, 1, figsize=(8.5, 7), sharex=True,
                         gridspec_kw={"hspace": 0.22})
for i, run in enumerate(runs):
    xs = [r["level"] for r in run["rows"]]
    axes[0].plot(xs, [r["out_tok_per_s"] for r in run["rows"]],
                 marker="o", color=SERIES[i], label=run["gpu"])
    axes[1].plot(xs, [(r["ttft_p99"] or 0) * 1000 for r in run["rows"]],
                 marker="o", color=SERIES[i], label=run["gpu"])
axes[0].set_ylabel("output tokens/s"); axes[0].set_title("throughput by card")
axes[0].set_xscale("log", base=2); axes[0].legend(loc="upper left")
axes[1].set_ylabel("TTFT p99 (ms)"); axes[1].set_xlabel("concurrency")
axes[1].legend(loc="upper left")
plt.show()

In [ ]:
# The chart that actually answers the question: what does a million tokens cost,
# at each card's best operating point *that still meets the SLO*?
from servlab.plots import bar_compare

SLO_TTFT = 1.0
best = []
for run in runs:
    ok = [r for r in run["rows"] if (r["ttft_p99"] or 9e9) <= SLO_TTFT]
    if not ok:
        print(f"{run['gpu']}: no operating point meets TTFT p99 <= {SLO_TTFT}s")
        continue
    b = max(ok, key=lambda r: r["out_tok_per_s"])
    key = next((k for k in nk.GPUS if k.replace("-", " ").split()[0] in run["gpu"]), None)
    usd = nk.GPUS[key].usd_per_hour if key else 0.0
    cost = usd / 3600 / b["out_tok_per_s"] * 1e6 if b["out_tok_per_s"] else float("inf")
    best.append({"gpu": run["gpu"], "concurrency": b["level"],
                 "tok_s": b["out_tok_per_s"], "usd_h": usd, "usd_per_m": cost})
    print(f"{run['gpu']:<28} best @ concurrency {b['level']:>3}: "
          f"{b['out_tok_per_s']:>7,.0f} tok/s  ${usd:.2f}/h  ->  ${cost:,.2f} per 1M tokens")

if best:
    bar_compare([b["gpu"] for b in best], [b["usd_per_m"] for b in best],
                title=f"$ per 1M output tokens (TTFT p99 <= {SLO_TTFT}s)",
                ylabel="USD", fmt="${:,.2f}")

## 4. The conclusions to be able to defend

Write your own answers; these are the questions.

1. **Which card is cheapest per token, and at what concurrency?** The answer
   changes with batch size, because cost per token is throughput per dollar and
   throughput depends on batch.
2. **Which card is cheapest per token *at your latency SLO*?** Usually a
   different answer — the big card's cheap point may sit at a concurrency whose
   p99 you cannot ship.
3. **Where does the cheap card stop being an option?** Not gradually: it stops
   when the model no longer fits. Memory is a cliff, not a slope.
4. **Did more FLOPs help?** For decode they should barely have. If your L4
   beat your T4 by much on decode throughput, find out what else changed —
   dtype, CUDA graphs, a different vLLM version.
5. **What would change the answer entirely?** Quantisation (lab 5) moves both
   the memory cliff and the bandwidth term at once; it can make the small card
   viable again.

### The honest caveat

Colab GPUs are shared and throttled, so absolute numbers are not
publication-grade. Ratios between cards measured the same day are still useful,
and the *shape* of every curve is right. If a number matters commercially,
re-measure on a dedicated box — and say exactly that when presenting it, because
being clear about what your benchmark does not prove is itself the signal.